# Minimal working example using BERT (`distBERT` model)

In [2]:
!pip install pandas torch transformers faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 36.9 MB/s eta 0:00:00


## Setup

In [3]:
import pandas as pd
import numpy as np
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel
import faiss

In [5]:
items = pd.read_csv("items.csv")
events = pd.read_csv("events.csv")
events["ts"] = pd.to_datetime(events["ts"])

## Pseudo data creation

## BERT encoder (What makes BERT work?)

In [6]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_NAME = "distilbert-base-uncased"  # for illustration, 66M model

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
encoder = AutoModel.from_pretrained(MODEL_NAME).to(DEVICE)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


**Evaluate the BERT encoder**

In [7]:
# ----------------------------
# 2) BERT encode helper
# ----------------------------

# turn off gradient calculation (no back propagation in using BERT)
@torch.no_grad()
def bert_embed(texts, max_len=128):
    batch = tokenizer(
        texts, padding=True, truncation=True, max_length=max_len, return_tensors="pt"
    )
    batch = {k: v.to(DEVICE) for k, v in batch.items()}
    out = encoder(**batch)
    cls = out.last_hidden_state[:, 0]          # first column [CLS]-like token for classification
    emb = F.normalize(cls, dim=-1)             # normalization
    return emb.cpu().numpy().astype("float32") # size (B, 768)


## Embedding items into vectors (for comparisons)

In [8]:
# ----------------------------
# 3) Offline job: item embeddings
# ----------------------------

items["text"] = items["title"] + ". " + items["description"]
item_vecs = bert_embed(items["text"].tolist())
item_id_list = items["item_id"].tolist()

# Build ANN index (inner product works with normalized vectors)
index = faiss.IndexFlatIP(item_vecs.shape[1])
index.add(item_vecs)


## What user-specific data are there?

In [9]:
# ----------------------------
# 4) Feature builder: user text from last N clicks
# ----------------------------
def build_user_text(user_id, events, items, N=3):
    hist = (events[events["user_id"] == user_id]
            .sort_values("ts")
            .tail(N)["item_id"]
            .tolist())
    if not hist:
        return "no history", set()
    text = items.set_index("item_id").loc[hist, "text"].tolist()
    return " ".join(text), set(hist)


## How to recommend "similar" item with BERT?

In [10]:
# ----------------------------
# 5) "What to recommend" function
# ----------------------------
def recommend(user_id, k=3):
    user_text, seen = build_user_text(user_id, events, items, N=3)
    u = bert_embed([user_text])  # (1, 768)
    scores, idx = index.search(u, k + len(seen))  # keep track of what was seen by the user
    recs = []
    for j in idx[0]:
        iid = item_id_list[j]
        if iid not in seen:
            recs.append(iid)
        if len(recs) == k:
            break
    return recs


In [11]:
for u in events["user_id"].unique():
  print(u, "->", recommend(u, k=5))

u1 -> ['i6', 'i13', 'i18', 'i1', 'i10']
u10 -> ['i19', 'i11', 'i7', 'i13', 'i5']
u2 -> ['i18', 'i11', 'i6', 'i17', 'i1']
u3 -> ['i8', 'i4', 'i18', 'i5', 'i6']
u4 -> ['i4', 'i18', 'i8', 'i6', 'i20']
u5 -> ['i12', 'i13', 'i1', 'i2', 'i18']
u6 -> ['i18', 'i16', 'i20', 'i5', 'i6']
u7 -> ['i8', 'i20', 'i4', 'i18', 'i6']
u8 -> ['i6', 'i7', 'i18', 'i3', 'i13']
u9 -> ['i5', 'i18', 'i6', 'i4', 'i1']


In [12]:

for u in events["user_id"].unique():
  rec_ids = recommend(u, k = 5)
  rec_titles = items.set_index("item_id").loc[rec_ids, "title"].tolist()
  print(f"{u} -> {rec_titles}")


u1 -> ['Yoga Mat', 'Wireless Earbuds', 'Portable Blender', 'Noise Cancelling Headphones', 'Hiking Backpack']
u10 -> ['Cycling Gloves', 'Air Fryer', 'Stainless Steel Water Bottle', 'Wireless Earbuds', 'Fitness Smartwatch']
u2 -> ['Portable Blender', 'Air Fryer', 'Yoga Mat', 'Smart Home Speaker', 'Noise Cancelling Headphones']
u3 -> ['Science Fiction Novel', 'Vegetarian Cookbook', 'Portable Blender', 'Fitness Smartwatch', 'Yoga Mat']
u4 -> ['Vegetarian Cookbook', 'Portable Blender', 'Science Fiction Novel', 'Yoga Mat', 'Personal Finance Book']
u5 -> ['Desk Lamp', 'Wireless Earbuds', 'Noise Cancelling Headphones', 'Mechanical Keyboard', 'Portable Blender']
u6 -> ['Portable Blender', 'Mystery Thriller Paperback', 'Personal Finance Book', 'Fitness Smartwatch', 'Yoga Mat']
u7 -> ['Science Fiction Novel', 'Personal Finance Book', 'Vegetarian Cookbook', 'Portable Blender', 'Yoga Mat']
u8 -> ['Yoga Mat', 'Stainless Steel Water Bottle', 'Portable Blender', 'Running Shoes', 'Wireless Earbuds']
u9